In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
cloud_pass = os.getenv("cloud_pass")


In [ ]:
import os
import pymysql

# Thông tin kết nối tới database Aiven (thay thế bằng thông tin thực tế của bạn)
db_config = {
    'host': os.getenv('MYSQL_HOST'),
    'port': int(os.getenv('MYSQL_PORT', '3306')),
    'db': os.getenv('MYSQL_DB'),
    'user': os.getenv('MYSQL_USER'),
    'password': os.getenv('MYSQL_PASSWORD'),
    'ssl': {'ssl': {}}
}


In [ ]:

# Kết nối và khởi tạo database
try:
    conn = pymysql.connect(**db_config)
    conn.autocommit = True
    cur = conn.cursor()
        # Xóa toàn bộ các bảng trong database (cẩn thận khi sử dụng!)
    tables = ["attendance", "grades", "courses_summary", "student_profile"]
    for table in tables:
        cur.execute(f"DROP TABLE IF EXISTS {table}")
    conn.commit()
    print("✅ Đã xóa toàn bộ các bảng trong database")
    # Ví dụ: Tạo bảng users
    queries = [
        """
        CREATE TABLE IF NOT EXISTS student_profile (
            roll_number VARCHAR(50) PRIMARY KEY,
            full_name VARCHAR(255),
            date_of_birth DATE,
            gender VARCHAR(10),
            id_card_number VARCHAR(20),
            home_address TEXT,
            phone_number VARCHAR(20),
            email_address VARCHAR(255),
            id_date_of_issue DATE,
            id_place_of_issue VARCHAR(255),
            parent_full_name VARCHAR(255),
            parent_phone_number VARCHAR(20),
            parent_address TEXT,
            parent_email VARCHAR(255),
            parent_job VARCHAR(100),
            parent_workplace VARCHAR(255),
            old_roll_number VARCHAR(50),
            member_code VARCHAR(50),
            enrollment_date DATE,
            study_mode VARCHAR(50),
            current_status VARCHAR(100),
            current_term_number INT,
            major VARCHAR(100),
            curriculum VARCHAR(100),
            capstone_project TEXT,
            main_class VARCHAR(50),
            specialization VARCHAR(100),
            account_balance VARCHAR(255),
            previous_major VARCHAR(100),
            decision_graduate_check VARCHAR(100),
            is_full_time_student BOOLEAN,
            full_time_confirmed_date DATE,
            is_scholarship_student BOOLEAN,
            valid_study_period VARCHAR(100),
            training_type VARCHAR(100),
            decision_dropout VARCHAR(100),
            decision_transfer_campus VARCHAR(100),
            decision_academic_leave VARCHAR(100),
            decision_graduation VARCHAR(100),
            decision_rejoin VARCHAR(100),
            destination_after_study VARCHAR(100),
            start_term VARCHAR(50)
        )
        """,

        """
        CREATE TABLE IF NOT EXISTS courses_summary (
            course_code VARCHAR(50),
            term VARCHAR(50),
            course_name VARCHAR(255),
            avg_score FLOAT,
            status VARCHAR(50),
            summary TEXT,
            PRIMARY KEY (course_code, term)
        )
        """,

        """
        CREATE TABLE IF NOT EXISTS grades (
            student_id VARCHAR(50),
            course_code VARCHAR(50),
            term VARCHAR(50),
            item VARCHAR(100),
            category VARCHAR(50),
            weight VARCHAR(20),
            value VARCHAR(20),
            PRIMARY KEY (student_id, course_code, item)
        )
        """,

        """
        CREATE TABLE IF NOT EXISTS attendance (
            student_id VARCHAR(50),
            course_code VARCHAR(50),
            term VARCHAR(50),
            date VARCHAR(50),
            slot VARCHAR(50),
            room VARCHAR(50),
            lecturer VARCHAR(100),
            `group` VARCHAR(50),
            status VARCHAR(50),
            comment TEXT,
            PRIMARY KEY (student_id, course_code, date, slot)
        )
        """
    ]

    for query in queries:
            cur.execute(query)
    conn.commit()
    print("Database và bảng đã được khởi tạo thành công.")
except Exception as e:
    print("Lỗi khi khởi tạo database:", e)

In [ ]:
cur.execute("DELETE FROM attendance")
cur.execute("DELETE FROM grades")
cur.execute("DELETE FROM courses_summary")
cur.execute("DELETE FROM student_profile")
conn.commit()
print("✅ Đã xóa toàn bộ dữ liệu trong tất cả các bảng")

In [ ]:
# import numpy as np

# def nan_to_none(x):
#     if pd.isna(x):
#         return None
#     return x

# for _, row in df_profile.iterrows():
#     cur.execute("""
#     INSERT IGNORE INTO student_profile (
#         full_name, date_of_birth, gender, id_card_number, home_address,
#         phone_number, email_address, id_date_of_issue, id_place_of_issue,
#         parent_full_name, parent_phone_number, parent_address, parent_email,
#         parent_job, parent_workplace, roll_number, old_roll_number, member_code,
#         enrollment_date, study_mode, current_status, current_term_number, major,
#         curriculum, capstone_project, main_class, specialization, account_balance,
#         previous_major, decision_graduate_check, is_full_time_student,
#         full_time_confirmed_date, is_scholarship_student, valid_study_period,
#         training_type, decision_dropout, decision_transfer_campus,
#         decision_academic_leave, decision_graduation, decision_rejoin,
#         destination_after_study, start_term
#     ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
#               %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
#               %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
#               %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
#               %s, %s)
#     """, tuple(nan_to_none(row[col]) for col in [
#         "full_name", "date_of_birth", "gender", "id_card_number", "home_address",
#         "phone_number", "email_address", "id_date_of_issue", "id_place_of_issue",
#         "parent_full_name", "parent_phone_number", "parent_address", "parent_email",
#         "parent_job", "parent_workplace", "roll_number", "old_roll_number", "member_code",
#         "enrollment_date", "study_mode", "current_status", "current_term_number", "major",
#         "curriculum", "capstone_project", "main_class", "specialization", "account_balance",
#         "previous_major", "decision_graduate_check", "is_full_time_student",
#         "full_time_confirmed_date", "is_scholarship_student", "valid_study_period",
#         "training_type", "decision_dropout", "decision_transfer_campus",
#         "decision_academic_leave", "decision_graduation", "decision_rejoin",
#         "destination_after_study", "start_term"
#     ]))
# conn.commit()
# print("✅ Insert đầy đủ 44 cột dữ liệu student_profile vào MySQL")


In [ ]:
import os
import pandas as pd
import pymysql
import numpy as np
import hashlib

class FapSQLUploader:
    def __init__(self, csv_paths: dict, db_config: dict):
        self.csv_paths = csv_paths
        self.conn = pymysql.connect(
            host=db_config["host"],
            port=db_config["port"],
            user=db_config["user"],
            password=db_config["password"],
            database=db_config["db"],
            ssl={'ssl': {}} if db_config.get("ssl") else None,
            cursorclass=pymysql.cursors.DictCursor
        )
        self.cursor = self.conn.cursor()

    def nan_to_none(self, x):
        return None if pd.isna(x) else x

    def hash_row(self, row, cols):
        row_str = '|'.join(str(row[col]) for col in cols)
        return hashlib.md5(row_str.encode('utf-8')).hexdigest()

    def load_dataframes(self):
        self.df_profile = pd.read_csv(self.csv_paths["student_profile"])

        # CHUYỂN NGÀY VỀ YYYY-MM-DD
        if "date_of_birth" in self.df_profile.columns:
            self.df_profile["date_of_birth"] = pd.to_datetime(
                self.df_profile["date_of_birth"], dayfirst=True, errors="coerce"
            ).dt.strftime("%Y-%m-%d")

        if "id_date_of_issue" in self.df_profile.columns:
            self.df_profile["id_date_of_issue"] = pd.to_datetime(
                self.df_profile["id_date_of_issue"], dayfirst=True, errors="coerce"
            ).dt.strftime("%Y-%m-%d")

        if "full_time_confirmed_date" in self.df_profile.columns:
            self.df_profile["full_time_confirmed_date"] = pd.to_datetime(
                self.df_profile["full_time_confirmed_date"], dayfirst=True, errors="coerce"
            ).dt.strftime("%Y-%m-%d")

        self.df_attendance = pd.read_csv(self.csv_paths["attendance_reports"])
        self.df_grades = pd.read_csv(self.csv_paths["grade_details"])
        self.df_summaries = pd.read_csv(self.csv_paths["course_summaries"])


    def filter_changed_rows(self, df, table, primary_keys, data_columns):
        cols_to_select = ', '.join(set(primary_keys + data_columns))
        self.cursor.execute(f"SELECT {cols_to_select} FROM {table}")
        rows = self.cursor.fetchall()
        existing = {}

        for row in rows:
            key = tuple(row[pk] for pk in primary_keys)
            existing[key] = self.hash_row(row, data_columns)

        changed_rows = []
        for _, row in df.iterrows():
            key = tuple(row[pk] for pk in primary_keys)
            new_hash = self.hash_row(row, data_columns)
            if key not in existing or new_hash != existing[key]:
                changed_rows.append(row)

        return pd.DataFrame(changed_rows)

    def insert_grades(self):
        cols = ["student_id", "course_code", "term", "item", "category", "weight", "value"]
        pks = ["student_id", "course_code", "term", "item"]
        df = self.filter_changed_rows(self.df_grades, "grades", pks, cols)

        if not df.empty:
            values = [tuple(self.nan_to_none(row[col]) for col in cols) for _, row in df.iterrows()]
            self.cursor.executemany("""
                INSERT INTO grades (
                    student_id, course_code, term, item, category, weight, value
                ) VALUES (%s, %s, %s, %s, %s, %s, %s)
                ON DUPLICATE KEY UPDATE category=VALUES(category), weight=VALUES(weight), value=VALUES(value)
            """, values)
            self.conn.commit()
        print("✅ Insert xong grade_details")
        return df

    def insert_attendance_reports(self):
        cols = ["student_id", "course_code", "term", "date", "slot", "room", "lecturer", "status", "comment"]
        pks = ["student_id", "course_code", "term", "date", "slot"]
        df = self.filter_changed_rows(self.df_attendance, "attendance", pks, cols)

        if not df.empty:
            values = [tuple(self.nan_to_none(row[col]) for col in cols) for _, row in df.iterrows()]
            self.cursor.executemany("""
                INSERT INTO attendance (
                    student_id, course_code, term, date, slot, room,
                    lecturer, status, comment
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON DUPLICATE KEY UPDATE room=VALUES(room), lecturer=VALUES(lecturer), status=VALUES(status), comment=VALUES(comment)
            """, values)
            self.conn.commit()
        print("✅ Insert xong attendance_reports")
        return df

    def insert_student_profile(self):
        cols = list(self.df_profile.columns)
        pks = ["roll_number"]
        df = self.filter_changed_rows(self.df_profile, "student_profile", pks, cols)

        if not df.empty:
            values = [tuple(self.nan_to_none(row[col]) for col in cols) for _, row in df.iterrows()]
            placeholders = ', '.join(['%s'] * len(cols))
            update_clause = ', '.join([f'{col}=VALUES({col})' for col in cols if col not in pks])
            self.cursor.executemany(f"""
                INSERT INTO student_profile ({', '.join(cols)})
                VALUES ({placeholders})
                ON DUPLICATE KEY UPDATE {update_clause}
            """, values)
            self.conn.commit()
        print("✅ Insert xong student_profile")
        return df

    def insert_course_summaries(self):
        cols = list(self.df_summaries.columns)
        pks = ["course_code", "term"]
        df = self.filter_changed_rows(self.df_summaries, "courses_summary", pks, cols)

        if not df.empty:
            values = [tuple(self.nan_to_none(row[col]) for col in cols) for _, row in df.iterrows()]
            placeholders = ', '.join(['%s'] * len(cols))
            update_clause = ', '.join([f'{col}=VALUES({col})' for col in cols if col not in pks])
            self.cursor.executemany(f"""
                INSERT INTO courses_summary ({', '.join(cols)})
                VALUES ({placeholders})
                ON DUPLICATE KEY UPDATE {update_clause}
            """, values)
            self.conn.commit()
        print("✅ Insert xong course_summaries")
        return df

if __name__ == "__main__":
    csv_paths = {
    }

    db_config = {
        'host': os.getenv('MYSQL_HOST'),
        'port': int(os.getenv('MYSQL_PORT', '3306')),
        'db': os.getenv('MYSQL_DB'),
        'user': os.getenv('MYSQL_USER'),
        'password': os.getenv('MYSQL_PASSWORD'),
        'ssl': {'ssl': {}}
    }

    uploader = FapSQLUploader(csv_paths, db_config)
    uploader.load_dataframes()

    updated_profiles = uploader.insert_student_profile()
    # updated_grades = uploader.insert_grades()
    # updated_attendance = uploader.insert_attendance_reports()
    # updated_summaries = uploader.insert_course_summaries()

    # Bạn có thể pass các `updated_*` DataFrame này vào bước embedding → đẩy lên Qdrant

In [ ]:
# Tải lại toàn bộ dữ liệu từ các bảng trong database về DataFrame và hiển thị
df_db_profile = pd.read_sql("SELECT * FROM student_profile", conn)
df_db_attendance = pd.read_sql("SELECT * FROM attendance", conn)
df_db_grades = pd.read_sql("SELECT * FROM grades", conn)
df_db_courses_summary = pd.read_sql("SELECT * FROM courses_summary", conn)

print("student_profile:")
display(df_db_profile)
print("attendance:")
display(df_db_attendance)
print("grades:")
display(df_db_grades)
print("courses_summary:")
display(df_db_courses_summary)